In [1]:
from dotenv import load_dotenv
import os

import snowflake.connector

load_dotenv()

account= os.getenv("SNOWFLAKE_ACCOUNT")
user= os.getenv("SNOWFLAKE_USER")
password= os.getenv("SNOWFLAKE_PASSWORD")
warehouse= os.getenv("SNOWFLAKE_WAREHOUSE")
database= os.getenv("SNOWFLAKE_DATABASE")

connection= snowflake.connector.connect(account= account, user= user, password= password, warehouse= warehouse, database= database)

print(connection)
print("Connection successful:", connection.account)

Connection successful: ui48522


In [2]:
import pandas as pd

boc_rates= pd.read_csv("../data/processed/boc_rates_monthly.csv")
crea_monthly= pd.read_csv("../data/processed/crea_monthly_benchmark.csv")

boc_rates.shape

(102, 10)

In [3]:
boc_rates.dtypes

date                            object
mortgage_arrears               float64
loan_to_income_ratio           float64
debt_service_ratio             float64
debt_service_ratio_RESL2       float64
mortgage_rate_5yrs_fixed       float64
mortgage_rate_5yrs_variable    float64
policy_rate                    float64
prime_rate                     float64
agg_debt_service_ratio         float64
dtype: object

In [5]:
cursor = connection.cursor()

cursor.execute('''CREATE TABLE IF NOT EXISTS HOUSING_AFFORDABILITY.RAW.boc_rates (
                    date DATE,
                    mortgage_arrears FLOAT,
                    loan_to_income_ratio FLOAT,
                    debt_service_ratio FLOAT,
                    debt_service_ratio_RESL2 FLOAT,
                    agg_debt_service_ratio FLOAT,
                    mortgage_rate_5yrs_fixed FLOAT,
                    mortgage_rate_5yrs_variable FLOAT,
                    policy_rate FLOAT,
                    prime_rate FLOAT       
                )''')

In [6]:
crea_monthly.shape

(784, 3)

In [7]:
crea_monthly.dtypes

Date                   object
City                   object
Composite_Benchmark     int64
dtype: object

In [8]:
cursor.execute('''CREATE TABLE IF NOT EXISTS HOUSING_AFFORDABILITY.RAW.crea_monthly (
                    Date DATE,
                    City TEXT,
                    Composite_Benchmark INT  
                )''')

In [9]:
from snowflake.connector import pandas_tools

boc_rates.columns = boc_rates.columns.str.upper()
cursor.execute("TRUNCATE TABLE HOUSING_AFFORDABILITY.RAW.BOC_RATES")

pandas_tools.write_pandas(connection, boc_rates, "BOC_RATES", schema= "RAW")

(True,
 1,
 102,
 [('snowpark_temp_stage_xm8e1wwgzv/file0.txt',
   'LOADED',
   102,
   102,
   1,
   0,
   None,
   None,
   None,
   None)])

In [10]:
crea_monthly.columns = crea_monthly.columns.str.upper()

cursor.execute("TRUNCATE TABLE HOUSING_AFFORDABILITY.RAW.CREA_MONTHLY")
pandas_tools.write_pandas(connection, crea_monthly, "CREA_MONTHLY", schema= "RAW")

(True,
 1,
 784,
 [('snowpark_temp_stage_fp6gde36n7/file0.txt',
   'LOADED',
   784,
   784,
   1,
   0,
   None,
   None,
   None,
   None)])

In [11]:
cursor.execute('SELECT * FROM HOUSING_AFFORDABILITY.RAW.BOC_RATES LIMIT 5')
boc_test = cursor.fetchall()
boc_test

[(datetime.date(2018, 1, 31),
  0.2,
  280.79,
  16.9,
  None,
  16.9,
  3.19,
  2.55,
  1.25,
  3.45),
 (datetime.date(2018, 2, 28),
  None,
  None,
  None,
  None,
  None,
  3.24,
  2.5,
  1.25,
  3.45),
 (datetime.date(2018, 3, 31),
  None,
  None,
  None,
  None,
  None,
  3.24,
  2.5,
  1.25,
  3.45),
 (datetime.date(2018, 4, 30),
  0.2,
  267.87,
  16.17,
  None,
  16.17,
  3.24,
  2.5,
  1.25,
  3.45),
 (datetime.date(2018, 5, 31),
  None,
  None,
  None,
  None,
  None,
  3.34,
  2.45,
  1.25,
  3.45)]

In [12]:
connection.close()